# Action and observation distributions
What the agent perceives and how its actions evolve over training. Includes sample trajectories.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

H5_PATH = "campaign_analysis.h5"
OBS_NAMES = ["fInf", "P_int", "Egamma", "gamma", "phi", "z", "maxOv", "t", "prev_aP", "prev_aS"]
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

f = h5py.File(H5_PATH, 'r')
act = f['actions']
obs = f['observations']
traj = f['trajectories']
rounds = act['rounds'][:]
print(f"Traj rounds: {rounds[0]}–{rounds[-1]}  ({len(rounds)} sampled)")

In [ ]:
# ---- action histogram waterfalls ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, key_e, key_c, title in [
    (ax1, 'aP_hist_edges', 'aP_counts', 'aP (pressure action)'),
    (ax2, 'aS_hist_edges', 'aS_counts', 'aS (shear action)'),
]:
    edges  = act[key_e][:]
    counts = act[key_c][:]   # (T, A)
    im = ax.imshow(
        counts, aspect='auto', origin='lower',
        extent=[edges[0], edges[-1], rounds[0], rounds[-1]],
        cmap='plasma', interpolation='nearest'
    )
    plt.colorbar(im, ax=ax, label='normalised density')
    ax.set_xlabel(title); ax.set_ylabel('training round')
    ax.set_title(f'{title} distribution')

fig.tight_layout()
fig.savefig('action_waterfalls.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- action mean ± std vs round ----
fig, ax = plt.subplots(figsize=(9, 3))
for key, label, c in [('aP', 'aP', 'steelblue'), ('aS', 'aS', 'tomato')]:
    mu  = act[f'{key}_mean'][:]
    sig = act[f'{key}_std'][:]
    ax.fill_between(rounds, mu - sig, mu + sig, alpha=0.2, color=c)
    ax.plot(rounds, mu, lw=1.5, color=c, label=label)
ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
ax.set_xlabel('round'); ax.set_ylabel('action value')
ax.set_title('Action mean ± std')
ax.legend()
fig.tight_layout()
fig.savefig('action_mean_std.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- observation distributions at 3 key stages ----
n = len(rounds)
stage_idx   = [0, n // 2, n - 1]
stage_labels = [f'early (r={rounds[i]})' for i in stage_idx]
stage_colors = ['steelblue', 'goldenrod', 'tomato']

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for j, name in enumerate(OBS_NAMES):
    ax = axes.ravel()[j]
    edges  = obs[f'{name}_edges'][:]
    counts = obs[f'{name}_counts'][:]   # (T, O)
    centres = 0.5 * (edges[:-1] + edges[1:])
    for k, (i, lbl, c) in enumerate(zip(stage_idx, stage_labels, stage_colors)):
        ax.plot(centres, counts[i], color=c, lw=1.2, label=lbl)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel('value', fontsize=7)
    if j == 0:
        ax.legend(fontsize=6)

fig.suptitle('Observation distributions: early / mid / late', fontsize=11)
fig.tight_layout()
fig.savefig('obs_distributions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- episode-level stats from trajectories ----
traj_rounds = traj['rounds'][:]
ep_lens, phi_means, dphi_means = [], [], []

for r in traj_rounds:
    rg = traj[f'round_{r:04d}']
    steps    = rg['steps'][:].astype(float)
    phi      = rg['phi'][:]
    phi_null = rg['phi_null'][:]
    ep_lens.append(np.nanmean(steps))
    phi_means.append(np.nanmean(phi))
    dphi_means.append(np.nanmean(phi - phi_null))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(traj_rounds, ep_lens, '.-', ms=4)
axes[0].set_title('Mean episode length (steps)'); axes[0].set_xlabel('round')

axes[1].plot(traj_rounds, phi_means, '.-', ms=4, color='steelblue')
axes[1].set_title('Mean φ of jammed states'); axes[1].set_xlabel('round')

axes[2].plot(traj_rounds, dphi_means, '.-', ms=4, color='tomato')
axes[2].axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
axes[2].set_title('Mean φ − φ_null'); axes[2].set_xlabel('round')

fig.tight_layout()
fig.savefig('episode_stats.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- sample reward trajectory from latest round ----
latest_r = traj_rounds[-1]
rg = traj[f'round_{latest_r:04d}']
rew    = rg['rew'][:]
ep_ptr = rg['ep_ptr'][:]

# pick the first episode
t0, t1 = int(ep_ptr[0]), int(ep_ptr[1])
ep_rew = rew[t0:t1]
ep_act = rg['act'][t0:t1]  # (T, 2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax1.plot(ep_rew, color='steelblue'); ax1.set_ylabel('reward')
ax1.set_title(f'Sample episode — round {latest_r}')
ax2.plot(ep_act[:, 0], label='aP'); ax2.plot(ep_act[:, 1], label='aS')
ax2.set_ylabel('action'); ax2.set_xlabel('timestep'); ax2.legend()
fig.tight_layout()
fig.savefig('sample_trajectory.pdf', bbox_inches='tight')
plt.show()